# ISOM 839 · Session 10 — You Take the Wheel: Markov Decision Processes

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

In Session 9 we *watched* customers move from state to state. Tonight we **choose what to do** in every state, knowing that today's choice changes tomorrow. That is a **Markov decision process** (MDP), and the tool that solves it is the **Bellman equation**.

### 🗺️ Tonight's road map: see it on a slide, then do it here

| Part | What you do | Time |
|---|---|---|
| **0 · Setup** | Run one cell | 1 min |
| **1 · Five words** | Who should get a \$4 gift? Write the problem down. | 10 min |
| **2 · Bellman** | One step by hand, then value iteration. Check every playbook. | 15 min |
| **3 · Run the shop** | Your code writes the ordering playbook (and answers the class vote) | 15 min |
| **4 · Turn the knobs** | Which costs move the playbook, and which don't? | 8 min |
| **Homework #7 · Option A** | A new shop where shelf space costs money | at home |

### How to use this notebook
| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter**, no typing needed |
| 🤔 **Bet first** | say your guess out loud *before* you run the next cell |
| ✅ **What you should see** | if your output matches, move on |
| ✏️ **Your turn** | the only places you change anything |

---
## Part 0 — Setup

In [ ]:
# @title ▶️ Run me: setup (no typing needed)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
np.set_printoptions(precision=2, suppress=True)
pd.set_option('display.precision', 2)

def card(title, rows, color='#0a66c2'):
    # a small result card: rows = [(label, value), ...]
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:720px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready!')

**✅ What you should see:** `✅ Ready!`

---
## Part 1 — Five words, and the gift problem

> 🎯 **In this part:** you write a decision problem down as data, and meet the manager who only looks at this month.

Every decision over time has the same five pieces:

| Word | Plain English | In the gift problem |
|---|---|---|
| **State** | Where am I right now? | the customer is 😊 **Happy** or 😟 **Unhappy** |
| **Action** | What can I do? | **leave** them alone, or send a \$4 **gift** |
| **Chance** | What happens next? | next month they may switch mood. The odds **depend on my action** |
| **Reward** | What is this month worth? | Happy pays \$10, Unhappy pays \$2, minus \$4 if I sent a gift |
| **Policy** | What do I do in *every* state? | a playbook: "Happy → ?, Unhappy → ?" **This is what we solve for.** |

State and chance you met in Session 9. **Action, reward and policy are new.**

### Step 1.1 — Write the problem down

A Markov chain has **one** table. A decision problem has **one table per action**: the gift changes the odds.

In [ ]:
# @title ▶️ Run me: the gift problem as data
states  = ['Happy', 'Unhappy']
actions = ['leave', 'gift']

P = {   #            to Happy  to Unhappy
    'leave': np.array([[0.8,      0.2],      # from Happy
                       [0.2,      0.8]]),    # from Unhappy
    'gift':  np.array([[0.9,      0.1],      # from Happy
                       [0.6,      0.4]]),    # from Unhappy  <- the gift helps most here
}
pays      = np.array([10, 2])   # what a Happy / Unhappy customer pays each month
GIFT_COST = 4

def reward(s, a, gift_cost=GIFT_COST):
    # this month's cash: what the customer pays, minus the gift if we sent one
    return pays[s] - (gift_cost if a == 'gift' else 0)

for a in actions:
    print(f"If we {a}:")
    print(pd.DataFrame(P[a], index=states, columns=states), '\n')
    assert np.allclose(P[a].sum(axis=1), 1), 'A row does not add up to 1!'
print('Every row adds to 1 ✓')

**✅ What you should see:** two 2 × 2 tables, then `Every row adds to 1 ✓`.

Read the key number out loud: *"An unhappy customer who gets a gift has a **60%** chance of being happy next month. Without it, only **20%**."*

### Step 1.2 — The one-month manager

Imagine a manager who only looks at **this month's cash**. What would she do?

**🤔 Bet first:** will she ever send a gift?

In [ ]:
# @title ▶️ Run me: the manager who only sees this month
for s, name in enumerate(states):
    cash = {a: reward(s, a) for a in actions}
    pick = max(cash, key=cash.get)
    print(f"{name:8s}  leave ${cash['leave']:>3}   gift ${cash['gift']:>3}   ->  she picks: {pick}")

**✅ What you should see:** she picks **leave** for both (\$10 beats \$6, and \$2 beats −\$2). The gift "costs \$4", so she never sends one.

She is ignoring **next month**. The gift is not about this month; it changes where the customer will be. To see that, we need to look ahead. That's what Bellman does in Part 2.

### Step 1.3 — The patience dial γ (gamma)

A dollar next month counts a little less than a dollar today. **γ = 0.9** means next month's dollar counts as 90¢, the month after as 0.9 × 0.9 = 81¢, and so on.

What is a customer who pays **\$10 every month, forever** worth today?

In [ ]:
# @title ▶️ Run me: $10 a month forever, discounted
gamma = 0.9
total = sum(10 * gamma**t for t in range(1000))     # month 0, 1, 2, ... (1,000 months is "forever")
print(f'Added up month by month: ${total:.2f}')
print(f'Shortcut 10 / (1 - gamma):  ${10 / (1 - gamma):.2f}')

**✅ What you should see:** both lines say **\$100.00**.

A handy rule: with discount γ, a business looks about **1 ÷ (1 − γ)** steps ahead. γ = 0.9 → about 10 months.

### ✏️ Your turn (1 minute)
Change `gamma` to **0.95** and re-run. *(✅ \$200: more patience, more value. Then set it back to 0.9.)*

---
## Part 2 — The Bellman equation

> 🎯 **In this part:** you put a **price tag** on every situation. Once every situation has a price tag, deciding is easy: for each choice, take *cash now + the price tag of where it lands you*, and pick the biggest. The Bellman equation is the rule that makes the price tags agree with each other.

> **V(state) = best over choices [ reward now + γ × average V(next state) ]**

In plain English: *how good is it to be here? The best of: what I get now, plus what my next spot is worth (a little discounted, averaged over where chance may take me).*

### Step 2.1 — One Bellman step by hand

Suppose someone already told us what each state is worth: Happy **\$73.66**, Unhappy **\$59.02**. Should we gift an unhappy customer?

In [ ]:
# @title ▶️ Run me: one Bellman step for an Unhappy customer
V_given = np.array([73.66, 59.02])   # value of Happy, value of Unhappy
gamma = 0.9
s = 1                                 # Unhappy

for a in actions:
    future = P[a][s] @ V_given        # average value of next month: odds x values
    q = reward(s, a) + gamma * future
    print(f"{a:5s}: ${reward(s, a):>3} now + 0.9 x ${future:.2f} later = ${q:.2f}")

**✅ What you should see:** `leave … = $57.75` and `gift … = $59.02`.

The gift loses \$4 **today** but wins about **\$1.27 overall**: it buys a better tomorrow. And notice: \$59.02 is exactly the value we started with for Unhappy. **The right values agree with themselves.** That suggests how to find them: guess, apply Bellman, repeat.

### Step 2.2 — Value iteration: guess, apply Bellman, repeat

Start with V = 0 everywhere. Each **sweep** applies the Bellman step to every state, using the previous sweep's values. Stop when nothing changes.

A nice way to read it: **after k sweeps, V is the best you can do if the world ended k months from now.** Sweep 1 = the one-month manager. Sweep 150 = a manager who sees (practically) forever.

**🤔 Bet first:** at which sweep will the unhappy customer start getting a gift?

In [ ]:
# @title ▶️ Run me: value iteration on the gift problem
def solve_gift(gamma=0.9, gift_cost=GIFT_COST, show=True, tol=1e-6):
    V = np.zeros(2)
    for sweep in range(1, 10_000):
        V_new, policy = np.zeros(2), []
        for s in range(2):
            q = {a: reward(s, a, gift_cost) + gamma * P[a][s] @ V for a in actions}   # Bellman, both choices
            best = max(q, key=q.get)
            V_new[s] = q[best]; policy.append(best)
        change = np.max(np.abs(V_new - V))
        V = V_new
        if show and (sweep <= 4 or change < tol):
            print(f"sweep {sweep:3d}:  V(Happy) ${V[0]:6.2f}   V(Unhappy) ${V[1]:6.2f}   Happy -> {policy[0]:5s}   Unhappy -> {policy[1]}")
        if show and sweep == 4:
            print('   ...')
        if change < tol:
            return V, policy, sweep

V_gift, policy_gift, n = solve_gift()
card('The Bellman answer', [('Sweeps until nothing changed', n),
                            ('Value of a Happy customer', f'${V_gift[0]:.2f}'),
                            ('Value of an Unhappy customer', f'${V_gift[1]:.2f}'),
                            ('Playbook', f'Happy → {policy_gift[0]},  Unhappy → {policy_gift[1]}')])

**✅ What you should see:** sweeps 1 and 2 say **leave** for both (the one-month and two-month managers). At **sweep 3** the Unhappy column flips to **gift**. It converges after **151 sweeps**: Happy **\$73.66**, Unhappy **\$59.02**. Playbook: *leave happy customers alone, gift the unhappy ones.*

### 📈 See it: the price tags climb, and the answer flips at sweep 3

In [ ]:
# @title ▶️ Run me: price tags sweep by sweep
V, tags, gift_unhappy = np.zeros(2), [], []
for sweep in range(1, 152):
    q = {(s, a): reward(s, a) + 0.9 * P[a][s] @ V for s in range(2) for a in actions}
    V = np.array([max(q[s, 'leave'], q[s, 'gift']) for s in range(2)])
    tags.append(V.copy()); gift_unhappy.append(q[1, 'gift'] > q[1, 'leave'])

tags = np.array(tags)
plt.figure(figsize=(8, 3.6))
plt.plot(range(1, 152), tags[:, 0], label='Happy', color='tab:green')
plt.plot(range(1, 152), tags[:, 1], label='Unhappy', color='tab:orange')
flip = gift_unhappy.index(True) + 1
plt.axvline(flip, color='gray', ls='--'); plt.text(flip + 2, 5, f'sweep {flip}: start gifting the unhappy', color='gray')
plt.xlim(0, 80); plt.xlabel('sweep (= months the manager looks ahead)'); plt.ylabel('price tag V ($)'); plt.legend(); plt.title('Value iteration on the gift problem (first 80 sweeps)')
plt.show()

**✅ What you should see:** two curves that climb fast and then flatten at **\$73.66** and **\$59.02**, and a dashed line at **sweep 3**. Each sweep adds one more month of looking ahead; the far future adds less and less because of γ.

### Step 2.3 — Don't trust it? Check every playbook

With 2 states and 2 actions there are only 2 × 2 = 4 playbooks. We can score each one directly: same loop, but **no "best over choices"**, just follow the playbook.

In [ ]:
# @title ▶️ Run me: score all 4 playbooks the brute-force way
def score_playbook(playbook, gamma=0.9, gift_cost=GIFT_COST):
    V = np.zeros(2)
    for _ in range(2000):   # follow the playbook, no max
        V = np.array([reward(s, playbook[s], gift_cost) + gamma * P[playbook[s]][s] @ V for s in range(2)])
    return V

rows = []
for a_happy in actions:
    for a_unhappy in actions:
        V = score_playbook([a_happy, a_unhappy])
        rows.append({'Happy does': a_happy, 'Unhappy does': a_unhappy,
                     'value of a Happy customer': V[0], 'value of an Unhappy customer': V[1]})
display(pd.DataFrame(rows).sort_values('value of a Happy customer', ascending=False).round(2))

**✅ What you should see:** the top row is **leave / gift** at **\$73.66**, the same answer value iteration found. *Gift nobody* scores \$68.70. *Gift everyone* is worst at about **\$50.14**: happy customers mostly stay happy anyway, so their gifts are wasted.

Same lesson as Session 9's \$2 twist: **targeted help beats perks for everyone.** Brute force works here because there are only 4 playbooks. The shop in Part 3 has **5,598,720**. Bellman doesn't care.

### ✏️ Your turn (3 minutes)
What if the gift were cheaper or pricier? The cell tries several prices. Find where the answer flips.

In [ ]:
# @title ✏️ Your turn: change the prices in the list, then run
for price in [0.5, 1, 2, 4, 6, 7, 8]:          # <- try your own gift prices
    _, pol, _ = solve_gift(gift_cost=price, show=False)
    print(f"gift costs ${price:4}:  Happy -> {pol[0]:5s}   Unhappy -> {pol[1]}")

**✅ What you should see:** at \$0.50 everyone gets a gift; from \$1 to \$6 only the unhappy do; at \$7 and up nobody does. The exact flip points are about **\$0.99** and **\$6.26**.

*The costs write the playbook.* Keep that sentence in mind for Part 4.

---
## Part 3 — Run the shop (tonight's lab)

> 🎯 **In this part:** your code writes a shop's ordering playbook from five numbers, and answers the class vote.

A tiny shop. Every morning you look at the shelf and decide how many units to order.

| Piece | Tonight's shop |
|---|---|
| **State** | units on the shelf this morning: 0 to 10 (the shelf holds 10) |
| **Action** | order 0 to 5 units (the truck holds 5). It arrives right away. You can't overfill the shelf. |
| **Chance** | 0, 1, 2 or 3 customers come, each 25% likely. Leftovers stay for tomorrow. |
| **Reward** | **+\$10** per sale, **−\$8** per unit bought, **−\$15** of goodwill per customer turned away |
| **Patience** | γ = 0.95 per day |

**The class vote:** 1 unit on the shelf, the truck is here. Order 0, 2, 4 or 5? You'll get the answer in Step 3.4.

### Step 3.1 — Describe the shop

All of it fits in one dictionary. Later we change one number at a time.

In [ ]:
# @title ▶️ Run me: tonight's shop
shop = dict(
    shelf     = 10,             # the shelf holds 0..10 units
    max_order = 5,              # the truck brings up to 5
    demand    = [0, 1, 2, 3],   # customers per day...
    prob      = [0.25, 0.25, 0.25, 0.25],   # ...each equally likely
    price     = 10,             # $ per sale
    cost      = 8,              # $ per unit bought
    penalty   = 15,             # $ of goodwill per customer turned away
    gamma     = 0.95,           # patience: a dollar tomorrow counts 95 cents
    hold      = 0,              # $ per unit left on the shelf overnight (0 tonight; Homework #7 uses it)
    fee       = 0,              # $ per truck delivery (0 tonight; Part 4 uses it)
)
assert abs(sum(shop['prob']) - 1) < 1e-9, 'Demand odds must add to 1'
print('Shop ready. Demand odds add to', sum(shop['prob']))

**✅ What you should see:** `Shop ready. Demand odds add to 1.0`

### Step 3.2 — One day in the shop

1 unit on the shelf, order 2, and 3 customers walk in. What happens?

In [ ]:
# @title ▶️ Run me: one day's money, and tomorrow's shelf
def one_day(shop, stock, order, customers):
    on_shelf = stock + order
    sold     = min(on_shelf, customers)
    missed   = customers - sold
    left     = on_shelf - sold                     # tomorrow starts with this
    money = (shop['price'] * sold - shop['cost'] * order - shop['penalty'] * missed
             - shop['hold'] * left - (shop['fee'] if order > 0 else 0))
    return money, left

money, tomorrow = one_day(shop, stock=1, order=2, customers=3)
print(f'Today: ${money}   Tomorrow starts with {tomorrow} units')

**✅ What you should see:** `Today: $14   Tomorrow starts with 0 units` (sold 3 × \$10 = \$30, bought 2 × \$8 = \$16, nobody turned away).

### Step 3.3 — Value iteration on the shop

Exactly the loop from Part 2. The only new thing: **chance**. For each shelf level and each order, we average over the 4 possible numbers of customers.

In [ ]:
# @title ▶️ Run me: value iteration writes the playbook
def scores(shop, V, stock):
    # Q(stock, order) for every allowed order: today's money + gamma x value of tomorrow's shelf, averaged over customers
    q = []
    for order in range(min(shop['max_order'], shop['shelf'] - stock) + 1):
        total = 0.0
        for customers, p in zip(shop['demand'], shop['prob']):
            money, left = one_day(shop, stock, order, customers)
            total += p * (money + shop['gamma'] * V[left])
        q.append(total)
    return np.array(q)

def value_iteration(shop, tol=1e-6, **changes):
    shop = {**shop, **changes}                 # e.g. value_iteration(shop, penalty=40)
    V = np.zeros(shop['shelf'] + 1)
    for sweep in range(1, 100_000):
        all_q  = [scores(shop, V, s) for s in range(shop['shelf'] + 1)]
        V_new  = np.array([q.max() for q in all_q])
        policy = np.array([int(q.argmax()) for q in all_q])
        change = np.max(np.abs(V_new - V))
        V = V_new
        if change < tol:
            return V, policy, sweep

V, policy, sweeps = value_iteration(shop)
print(f'Converged in {sweeps} sweeps\n')
display(pd.DataFrame({'order': policy, 'shelf after the truck': np.arange(11) + policy, 'value V': [f'${v:.2f}' for v in V]},
                     index=pd.Index(range(11), name='shelf this morning')))

plt.figure(figsize=(8, 3))
plt.bar(range(11), policy, color='orange')
plt.xticks(range(11)); plt.yticks(range(6)); plt.xlabel('units on the shelf this morning'); plt.ylabel('units to order')
plt.title('The playbook your code wrote'); plt.show()

**✅ What you should see:** `Converged in 292 sweeps`, and the order column **3, 2, 1, 0, 0, …, 0**. The "shelf after the truck" column is **3, 3, 3, 3** for the low shelves.

**Read it like a manager:** *"Top up to 3."* Three is the busiest possible day, so you never turn anyone away. Nobody told the code this rule; it found it from five numbers. Inventory theorists call it an **order-up-to** (or base-stock) rule.

Look at the value column too: an empty shelf is worth **\$48**, a shelf with 3 units **\$72**. Each of those first units is worth exactly **\$8**, the price you'd otherwise pay for it.

### 📈 See it: the shop's price tags fill in
The same loop, but we take snapshots after a few sweeps. Watch the **sweep 1** playbook: that's the manager who only sees today.

In [ ]:
# @title ▶️ Run me: snapshots of value iteration on the shop
V_k = np.zeros(11)
plt.figure(figsize=(8, 3.6))
for sweep in range(1, 293):
    all_q = [scores(shop, V_k, s) for s in range(11)]
    V_k = np.array([q.max() for q in all_q]); pol_k = [int(q.argmax()) for q in all_q]
    if sweep in (1, 2, 5, 20, 292):
        plt.plot(range(11), V_k, marker='o', label=f'after sweep {sweep}')
        print(f'sweep {sweep:3d}: fill up to {[s + a for s, a in enumerate(pol_k)][:3]} at shelf 0-2   empty shelf worth ${V_k[0]:7.2f}')
plt.xlabel('units on the shelf this morning'); plt.ylabel('price tag V ($)'); plt.legend(); plt.title('The price tags fill in')
plt.show()

**✅ What you should see:** after **sweep 1** the shop fills up to only **2** (a third unit might not sell *today*), and an empty shelf is worth **−\$7.25**. From **sweep 2** on it fills up to **3**: looking one more day ahead, an unsold unit isn't wasted, it sells tomorrow. The curves then rise until they settle at \$48 … \$120.

### Step 3.4 — The class vote answer: score every order at 1 unit

**🤔 Bet first:** 0, 2, 4 or 5?

In [ ]:
# @title ▶️ Run me: the scoreboard at 1 unit on the shelf
q1 = scores(shop, V, stock=1)
for order, q in enumerate(q1):
    flag = '   <- best' if order == q1.argmax() else ''
    print(f'order {order}: ${q:6.2f}  ' + '█' * int((q - 40) * 2) + flag)

**✅ What you should see:** **order 2 wins at \$56.00** (answer **B**). Order 0 scores \$43.75: it throws away \$12.25 of value, mostly in turned-away customers. Orders 3, 4, 5 are close (\$55.48, \$54.79, \$53.89) but pay for stock too early.

These scores have a name: **Q(state, action)**, the value of doing *this* in *this* state. You will meet Q again in reinforcement learning (Session 12), where an agent *learns* Q by trial and error instead of computing it.

### Step 3.5 — Watch the playbook run the shop for a month

Same 30 days of customers for two managers: the **playbook** and a **gut rule** ("we sell about 1.5 a day, so order 1 a day").

In [ ]:
# @title ▶️ Run me: 30 days in the shop
rng = np.random.default_rng(839)
days = rng.choice(shop['demand'], size=30, p=shop['prob'])

def run_month(rule, days, start=1):
    stock, log = start, []
    for d in days:
        order = min(rule(stock), shop['max_order'], shop['shelf'] - stock)
        money, left = one_day(shop, stock, order, d)
        log.append(dict(stock=stock, order=order, customers=d, turned_away=d - min(stock + order, d), money=money))
        stock = left
    return pd.DataFrame(log)

playbook = run_month(lambda s: policy[s], days)
gut      = run_month(lambda s: 1, days)
print('First 7 days with the playbook:'); display(playbook.head(7))
card('30 days, same customers', [('Playbook: customers turned away', int(playbook.turned_away.sum())),
                                 ('Playbook: total profit', f'${playbook.money.sum():,}' if playbook.money.sum() >= 0 else f'−${-playbook.money.sum():,}'),
                                 ('Gut rule "order 1 a day": turned away', int(gut.turned_away.sum())),
                                 ('Gut rule: total profit', f'${gut.money.sum():,}' if gut.money.sum() >= 0 else f'−${-gut.money.sum():,}')])

**✅ What you should see:** the playbook turns away **0** customers and makes money. The gut rule turns customers away (the exact count depends on the dice, but it's well above 0) and loses money. Ordering the *average* demand is a classic trap: on busy days you run out.

---
## Part 4 — Turn the knobs

> 🎯 **In this part:** you find out which costs actually change the playbook (and which don't), the same experiments as the class game and the slider slide.

`value_iteration(shop, penalty=40)` re-solves the shop with one number changed. Same code, new playbook.

### Step 4.1 — A premium brand: penalty \$15 → \$40

**🤔 Bet first:** a turned-away customer now costs much more. Does the playbook stock more?

In [ ]:
# @title ▶️ Run me: penalty $40
_, pol40, _ = value_iteration(shop, penalty=40)
print('penalty $15:', policy.tolist())
print('penalty $40:', pol40.tolist())

**✅ What you should see:** the **same** playbook.

Why? At 3 units you **never** turn anyone away. A bigger penalty for something that never happens changes nothing. It's like a constraint with slack in Session 3: its shadow price is zero. **A cost only moves the playbook when it is in a tug-of-war.**

### Step 4.2 — Every truck now costs \$5

A delivery fee is paid once per order, no matter how many units.

In [ ]:
# @title ▶️ Run me: a $5 truck fee
_, pol_fee, _ = value_iteration(shop, fee=5)
print('no fee: ', policy.tolist())
print('$5 fee: ', pol_fee.tolist())

def long_run(shop, policy):
    # Session 9's tool: the playbook turns the shop into a Markov chain. Read its steady state.
    n = shop['shelf'] + 1
    T = np.zeros((n, n))
    for s in range(n):
        for d, p in zip(shop['demand'], shop['prob']):
            T[s, one_day(shop, s, policy[s], d)[1]] += p
    x = np.zeros(n); x[0] = 1
    for _ in range(3000):
        x = x @ T
    profit = sum(x[s] * p * one_day(shop, s, policy[s], d)[0] for s in range(n) for d, p in zip(shop['demand'], shop['prob']))
    left   = sum(x[s] * p * one_day(shop, s, policy[s], d)[1] for s in range(n) for d, p in zip(shop['demand'], shop['prob']))
    trucks = 7 * sum(x[s] for s in range(n) if policy[s] > 0)
    return round(trucks, 2), round(left, 2), round(profit, 2)

fee_shop = {**shop, 'fee': 5}
display(pd.DataFrame([long_run(fee_shop, policy), long_run(fee_shop, pol_fee)],
                     index=['old habit: top up to 3 daily', 'new playbook'],
                     columns=['trucks per week', 'units on shelf at night', 'profit per day']))

**✅ What you should see:** the new playbook is **5, 5, 5, 0, 0, …**: *wait until the shelf is at 2 or fewer, then order a full truck.* In the table: trucks per week **5.25 → 2.10**, units on the shelf at night **1.5 → 3.5**, profit per day **−\$0.75 → +\$1.50**.

This "wait, then fill up" shape is the famous **(s, S) policy**; Herbert Scarf proved in 1960 that it is optimal when each order has a fixed fee.

**The manager's question:** the CFO sees fuller shelves and frowns. Your one-liner: *"We carry 2 extra units so we can skip 3 trucks a week. That turns a loss into about \$820 a year."*

### Step 4.3 — The tug-of-war: busy days and a holding cost

Now customers are **0 to 5** a day, and every unit left overnight costs **\$4** (rent, cash tied up, some goes stale). Running out and having too much now both cost money.

In [ ]:
# @title ▶️ Run me: busy days, $4 holding cost, two penalties
busy = dict(demand=[0, 1, 2, 3, 4, 5], prob=[1/6] * 6, hold=4)
for pen in [15, 40]:
    _, pol, _ = value_iteration(shop, penalty=pen, **busy)
    top_up = max(s + a for s, a in enumerate(pol) if a > 0)
    print(f'penalty ${pen}:  {pol.tolist()}   -> top up to {top_up}')

**✅ What you should see:** penalty \$15 → **top up to 4**. Penalty \$40 → **top up to 5**. Now the penalty matters, because it pulls against the holding cost.

### ✏️ Your turn: the patience dial
Change γ and watch the number of sweeps. Does the playbook change?

In [ ]:
# @title ✏️ Your turn: try other values of gamma
for g in [0.8, 0.95, 0.99]:          # <- try 0.5 or 0.999 too (0.999 takes a while)
    V_g, pol_g, n_g = value_iteration(shop, gamma=g)
    print(f'gamma {g}: {n_g:5d} sweeps   playbook {pol_g.tolist()}   empty shelf worth ${V_g[0]:.0f}')

**✅ What you should see:** **68** sweeps at 0.8, **292** at 0.95, **1,485** at 0.99. Same playbook every time; the values change a lot.

γ controls **how far ahead** the business looks, so more patience means more sweeps to be sure. In this shop it doesn't change *what* to do. (With a fee or a holding cost, it can. Try it!)

---
### 🌍 Bonus: where tables stop

Our shop has 11 states. A store with several products needs one state for every **combination** of shelf levels.

In [ ]:
# @title ▶️ Run me: the curse of dimensionality
for products in [1, 2, 5, 10]:
    print(f'{products:2d} products: {11**products:,} states')
print(f'1,000 products: a number with {len(str(11**1000)):,} digits')

**✅ What you should see:** 10 products already need **25,937,424,601** states, and 1,000 products a number with **1,042 digits**. Bellman called this the *curse of dimensionality*.

The fix is to stop storing V in a table and **learn a formula for it**, for example with a neural network. That is deep reinforcement learning (Session 12). Warren Powell would file tonight's method under **value function approximation (VFA)**, one of his four classes of policies.

---
### 🧰 Bonus: `solve_mdp()` for your final project

Any small decision problem, in the five steps from the slides: list the states, list the actions, write the odds (**one table per action**), write the money, pick γ. This function does the rest.

In [ ]:
# @title ▶️ Run me: a reusable solver for small MDPs
def solve_mdp(P, R, gamma, states=None, tol=1e-6):
    # P: dict action -> (n x n) table of odds (every row adds to 1)
    # R: dict action -> list of n rewards (cash now for that action in each state)
    # Returns the price tags V and the playbook as a table.
    acts = list(P); n = len(next(iter(P.values())))
    for a in acts:
        assert np.allclose(np.asarray(P[a]).sum(axis=1), 1), f'A row of the {a} table does not add up to 1'
    V = np.zeros(n)
    while True:
        Q = np.array([np.asarray(R[a]) + gamma * np.asarray(P[a]) @ V for a in acts])   # one row per action
        V_new = Q.max(axis=0)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    return pd.DataFrame({'best action': [acts[k] for k in Q.argmax(axis=0)], 'price tag V': V_new.round(2)},
                        index=states or range(n))

# Check: the gift problem gives the same answer as Part 2
display(solve_mdp(P, {'leave': [10, 2], 'gift': [6, -2]}, gamma=0.9, states=states))

**✅ What you should see:** Happy → **leave**, \$73.66 · Unhappy → **gift**, \$59.02. Same as Part 2.

**For your project:** replace `P`, `R` and `states` with your own. Churn with a win-back offer, a machine that wears out (repair now or wait?), a fare that can open or close: they all fit.

---
# Homework #7 · Option A — A shop where shelf space costs money 📦

**Pick one of the two Homework #7 options on Canvas: this one, or the pricing curve in the Session 8 notebook. Same points, same rubric.** Due **Wednesday, December 2, 7:40 PM**.

A different shop with a wider demand distribution, so the holding cost has room to matter. **State:** 0–12 units. **Action:** order 0–6. **Demand:** 0–5 units with probabilities 0.10, 0.20, 0.30, 0.20, 0.15, 0.05. **Price \$12, cost \$7, penalty \$6, γ = 0.95.** Deliveries arrive right away and there is no truck fee. The `value_iteration` function from Part 3 already has a `hold` knob: `value_iteration(hw_shop, hold=1)`.

1. **Baseline vs holding cost.** Run value iteration with no holding cost, then with **\$1 per unit of leftover inventory per night** (the real cost of tied-up cash and shelf space). Print the two playbooks side by side, with their top-up levels.
2. **How did the top-up level move, and why?** Two or three sentences. Which side of the tug-of-war got stronger? *(Part 4.3)*
3. **Sweep the holding cost from \$0 to \$4** in steps of \$0.50 and plot the top-up level against it.
4. **One paragraph for a store manager:** in plain English, what trade-off is the model balancing, and when would carrying "too much" stock actually be the right call?

In [ ]:
hw_shop = dict(shelf=12, max_order=6,
               demand=[0, 1, 2, 3, 4, 5], prob=[0.10, 0.20, 0.30, 0.20, 0.15, 0.05],
               price=12, cost=7, penalty=6, gamma=0.95, hold=0, fee=0)

def top_up_level(policy):
    # the shelf level the playbook fills up to (the most it ever has right after a truck)
    return max(s + a for s, a in enumerate(policy) if a > 0)

# TODO 1: value_iteration(hw_shop) and value_iteration(hw_shop, hold=1); print both playbooks and top_up_level() of each

# TODO 2: answer in a markdown cell below: which way the top-up level moved and why

# TODO 3: loop hold over np.arange(0, 4.01, 0.5); plot top_up_level against the holding cost

# TODO 4: the manager paragraph in a markdown cell below (and in the Canvas text box, starting with "Option A")

---
### Submission checklist (Option A)
- [ ] Both playbooks printed side by side, top-up levels stated
- [ ] The explanation of the move (which cost it trades against)
- [ ] The sweep plot
- [ ] The manager paragraph, pasted into the Canvas text box, starting with "Option A"

**Next class: Session 11 on December 2** (no class November 25, Thanksgiving). Deep learning meets optimization: predict, then decide. 🧠